# 02 — Model Training

**XGBoost + isotonic calibration** on engineered features.

Key design choices:
- `scale_pos_weight` handles 28:1 class imbalance — no SMOTE needed
- `eval_metric=aucpr` optimises PR-AUC directly (better than ROC for imbalanced data)
- Isotonic regression calibrates raw probabilities so P(fraud) is interpretable

Outputs:
- `models/fraud_xgb_calibrated.pkl`
- `data/ieee-fraud-detection/test_predictions.parquet`

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, fbeta_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

DATA_DIR  = Path("../../..") / "data" / "ieee-fraud-detection"
MODEL_DIR = Path("../../..") / "models"
MODEL_DIR.mkdir(exist_ok=True)


## 1. Define columns to drop

Drop IDs, raw strings already encoded into features, and high-cardinality fields that would leak or overfit.

In [ ]:
DROP_COLS = [
    "TransactionID", "TransactionDT",
    "card2", "card3", "card5",           # high-cardinality card metadata
    "addr1", "addr2",                    # raw address codes
    "P_emaildomain", "R_emaildomain",    # encoded via ratio/network features
    "DeviceInfo", "DeviceType",          # encoded via network features
    "card4", "card6", "ProductCD",       # raw strings
    "M1","M2","M3","M4","M5","M6","M7","M8","M9",     # match flags (string)
    "id_12","id_15","id_16","id_23","id_27","id_28",  # string identity cols
    "id_29","id_30","id_31","id_32","id_33","id_34",
    "id_35","id_36","id_37","id_38",
]

UI_FEATURES = [
    "TransactionAmt", "addr_age_days", "email_domain_mismatch",
    "new_acct_highval_express", "velocity_count_1h", "velocity_value_1h",
    "velocity_count_24h", "velocity_value_24h", "device_linkage_count",
    "has_identity", "dist1", "dist1_missing", "c_total",
]
print(f"Columns to drop: {len(DROP_COLS)}")


## 2. Load and prepare features

In [ ]:
df_full = pd.read_parquet(DATA_DIR / "train_engineered.parquet")
y = df_full["isFraud"].astype(np.int8)
drop = [c for c in DROP_COLS if c in df_full.columns] + ["isFraud"]
X = df_full.drop(columns=drop)

# Encode any remaining object columns as category codes
for col in X.select_dtypes("object").columns:
    X[col] = X[col].astype("category").cat.codes

print(f"Feature matrix: {X.shape[0]:,} rows  ×  {X.shape[1]} columns")
print(f"Fraud rate: {y.mean():.2%}")


## 3. Stratified train / calibration / test split

70% train · 10% calibration · 20% test. Calibration set is held out from training and used to fit the isotonic regressor.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_trainval, y_trainval, test_size=0.125, stratify=y_trainval, random_state=42
)   # 0.125 of 0.80 = 10% of total

print(f"Train : {len(X_train):,}")
print(f"Cal   : {len(X_cal):,}")
print(f"Test  : {len(X_test):,}")


## 4. Compute class weight

`scale_pos_weight = neg / pos` tells XGBoost to penalise missing a fraud case ~28× more than a false positive during training. This is equivalent to upsampling fraud without actually duplicating rows.

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = round(neg / pos)
print(f"Negative (legit): {neg:,}")
print(f"Positive (fraud): {pos:,}")
print(f"scale_pos_weight: {spw}  (ratio ≈ {neg/pos:.1f})")


## 5. Train XGBoost

`eval_metric=aucpr` monitors PR-AUC on the calibration set. `early_stopping_rounds=30` halts if no improvement for 30 rounds.

In [ ]:
xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=spw,
    eval_metric="aucpr",
    early_stopping_rounds=30,
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_cal, y_cal)],
    verbose=50,
)
print(f"\nBest iteration: {xgb.best_iteration}")


## 6. Isotonic calibration

XGBoost raw probabilities are well-ranked but not well-calibrated (not true P(fraud)). We fit an isotonic regression on the calibration set to map raw scores → calibrated probabilities.

> Note: `CalibratedClassifierCV(cv='prefit')` was removed in sklearn 1.2. We do this manually instead.

In [ ]:
raw_cal = xgb.predict_proba(X_cal)[:, 1]

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(raw_cal, y_cal)

# Compare raw vs calibrated on cal set
raw_test = xgb.predict_proba(X_test)[:, 1]
y_prob   = iso.predict(raw_test)

print(f"Raw score range  : {raw_test.min():.4f} – {raw_test.max():.4f}")
print(f"Calibrated range : {y_prob.min():.4f} – {y_prob.max():.4f}")


## 7. Evaluate on test set

In [ ]:
pr_auc = average_precision_score(y_test, y_prob)

# F-beta (β=2) rewards recall twice as much as precision
y_pred = (y_prob >= 0.5).astype(int)
fb2    = fbeta_score(y_test, y_pred, beta=2, zero_division=0)

print("=== Test Set Results ===")
print(f"  PR-AUC      : {pr_auc:.4f}  (primary metric)")
print(f"  F-beta β=2  : {fb2:.4f}  (at threshold=0.5)")
print()
print("Note: optimal threshold is not 0.5 — see notebook 03 for cost-based selection.")


## 8. Save model

In [ ]:
model_path = MODEL_DIR / "fraud_xgb_calibrated.pkl"
with open(model_path, "wb") as f:
    pickle.dump({"iso": iso, "xgb_base": xgb, "feature_names": list(X.columns)}, f)
print(f"Saved → {model_path}")


## 9. Save test predictions

In [ ]:
test_idx = X_test.index
ui_cols  = [c for c in UI_FEATURES if c in df_full.columns]
test_out = df_full.loc[test_idx, ["TransactionID", "isFraud"] + ui_cols].copy()
test_out["y_prob"] = y_prob

pred_path = DATA_DIR / "test_predictions.parquet"
test_out.to_parquet(pred_path, index=False)
print(f"Saved → {pred_path}  ({len(test_out):,} rows)")
test_out.head()
